In [1]:
# PHASE-01

import pandas as pd
import numpy as np

rng = np.random.default_rng(2025)

stores = pd.DataFrame({
    'StoreID': range(1, 9),
    'City': ['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai', 'Hyderabad', 'Kolkata', 'Jaipur'],
    'Region': ['West', 'North', 'South', 'West', 'South', 'South', 'East', 'North']
})

products = pd.DataFrame({
    'ProductID': range(101, 121),
    'Category': rng.choice(['Grocery', 'Electronics', 'Apparel', 'Home'], 20),
    'UnitCost': rng.uniform(50, 15000, 20).round(2)
})
products['UnitPrice'] = (products['UnitCost'] * rng.uniform(1.2, 1.8, 20)).round(2)

n_cust = 300
customers = pd.DataFrame({
    'CustID': range(1, n_cust + 1),
    'Name': [f'  Customer_{i} ' for i in range(1, n_cust + 1)],
    'SignupDate': pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 500, n_cust), unit='D'),
    'Segment': rng.choice(['regular', 'REGULAR', 'premium', 'PREMIUM', 'vip'], n_cust)
})

n_txn = 5000
transactions = pd.DataFrame({
    'TxnID': range(1, n_txn + 1),
    'CustID': rng.choice(range(1, n_cust + 15), n_txn),
    'StoreID': rng.choice(range(1, 9), n_txn),
    'ProductID': rng.choice(range(101, 121), n_txn),
    'Quantity': rng.choice([1, 2, 3, 4, 5, -1], n_txn, p=[0.3, 0.25, 0.2, 0.15, 0.05, 0.05]),
    'TxnDate': pd.to_datetime('2023-01-01') + pd.to_timedelta(rng.integers(0, 730, n_txn), unit='D'),
    'PaymentType': rng.choice(['Card', 'card', 'CASH', 'cash', 'UPI', 'upi'], n_txn)
})

transactions = pd.concat([transactions, transactions.sample(100, random_state=1)], ignore_index=True)

print(f"Stores:{stores.shape} Products:{products.shape} Customers:{customers.shape} Transactions:{transactions.shape}")

Stores:(8, 3) Products:(20, 4) Customers:(300, 4) Transactions:(5100, 7)


In [2]:
customers_clean = (
    customers
    .assign(
        Name=lambda x: x['Name'].str.strip(),
        Segment=lambda x: x['Segment'].str.lower()
    )
)

transactions_clean = (
    transactions
    .drop_duplicates(subset='TxnID')
    .query('Quantity > 0')
    .assign(PaymentType=lambda x: x['PaymentType'].str.lower())
)

dupes_removed = len(transactions) - len(transactions.drop_duplicates(subset='TxnID'))
invalid_qty_removed = len(transactions.drop_duplicates(subset='TxnID')) - len(transactions_clean)

print(f"Transactions: {len(transactions)} → {len(transactions_clean)} after cleaning")
print(transactions_clean['PaymentType'].value_counts())

invalid_cust = transactions_clean[~transactions_clean['CustID'].isin(customers_clean['CustID'])]
print(f"Transactions with unknown CustID: {len(invalid_cust)}")
print(sorted(invalid_cust['CustID'].unique()))

Transactions: 5100 → 4751 after cleaning
PaymentType
cash    1601
card    1580
upi     1570
Name: count, dtype: int64
Transactions with unknown CustID: 214
[np.int64(301), np.int64(302), np.int64(303), np.int64(304), np.int64(305), np.int64(306), np.int64(307), np.int64(308), np.int64(309), np.int64(310), np.int64(311), np.int64(312), np.int64(313), np.int64(314)]


In [3]:
master = (
    transactions_clean
    .merge(products, on='ProductID', how='left')
    .merge(stores, on='StoreID', how='left')
    .merge(customers_clean, on='CustID', how='left', indicator='cust_status')
    .assign(
        Revenue=lambda x: x['Quantity'] * x['UnitPrice'],
        Profit=lambda x: x['Quantity'] * (x['UnitPrice'] - x['UnitCost']),
        Segment=lambda x: x['Segment'].fillna('guest')
    )
)

print(f"Master table: {master.shape}")
print(master['cust_status'].value_counts())

master['Year'] = master['TxnDate'].dt.year
master['Month'] = master['TxnDate'].dt.to_period('M').astype(str)
master['Quarter'] = master['TxnDate'].dt.to_period('Q').astype(str)
master['DayOfWeek'] = master['TxnDate'].dt.day_name()

store_rev = master.groupby('StoreID')['Revenue'].transform('sum')
master['StoreRevRank'] = store_rev.rank(ascending=False, method='dense').astype(int)

master = master.sort_values('TxnDate')
master['CustCumRevenue'] = master.groupby('CustID')['Revenue'].cumsum()

Master table: (4751, 18)
cust_status
both          4537
left_only      214
right_only       0
Name: count, dtype: int64


In [4]:
monthly_rev = (
    master.groupby('Month')['Revenue']
    .sum()
    .to_frame()
    .assign(
        MoM_Growth=lambda x: x['Revenue'].pct_change() * 100,
        Rolling3Month=lambda x: x['Revenue'].rolling(3).mean()
    )
)
print(monthly_rev)

city_quarter = master.groupby(['City', 'Quarter']).agg(
    revenue=('Revenue', 'sum'),
    profit=('Profit', 'sum'),
    orders=('TxnID', 'count')
)
print(city_quarter.xs('2024Q4', level='Quarter').sort_values('revenue', ascending=False))

pivot_city_month = pd.pivot_table(
    master, values='Revenue', index='City',
    columns='Quarter', aggfunc='sum', fill_value=0, margins=True
)
print(pivot_city_month)

            Revenue  MoM_Growth  Rolling3Month
Month                                         
2023-01  3831461.74         NaN            NaN
2023-02  4748797.84   23.942197            NaN
2023-03  4966214.47    4.578351   4.515491e+06
2023-04  5446988.95    9.680904   5.054000e+06
2023-05  5849143.86    7.383068   5.420782e+06
2023-06  4844869.36  -17.169598   5.380334e+06
2023-07  4570229.70   -5.668670   5.088081e+06
2023-08  5402758.04   18.216335   4.939286e+06
2023-09  5634463.60    4.288653   5.202484e+06
2023-10  5001719.44  -11.229892   5.346314e+06
2023-11  4780004.11   -4.432782   5.138729e+06
2023-12  4793103.42    0.274044   4.858276e+06
2024-01  4788755.60   -0.090710   4.787288e+06
2024-02  4705466.20   -1.739270   4.762442e+06
2024-03  5055299.25    7.434610   4.849840e+06
2024-04  4471927.37  -11.539809   4.744231e+06
2024-05  5554819.86   24.215342   5.027349e+06
2024-06  5078990.60   -8.566061   5.035246e+06
2024-07  4740820.54   -6.658214   5.124877e+06
2024-08  4521

In [5]:
known = master[master['cust_status'] == 'both']
snapshot_date = known['TxnDate'].max() + pd.Timedelta(days=1)

rfm = known.groupby('CustID').agg(
    Recency=('TxnDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('TxnID', 'count'),
    Monetary=('Revenue', 'sum')
).reset_index()

rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4])

rfm['RFM_Score'] = (
    rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)
)

conditions = [rfm['RFM_Score'] >= 10, rfm['RFM_Score'] >= 7, rfm['RFM_Score'] >= 4]
choices = ['Champions', 'Loyal', 'At Risk']
rfm['CustomerSegment'] = np.select(conditions, choices, default='Lost')

print(rfm['CustomerSegment'].value_counts())
print(rfm.nlargest(5, 'Monetary')[['CustID', 'Recency', 'Frequency', 'Monetary', 'CustomerSegment']])

CustomerSegment
Loyal        107
At Risk       96
Champions     77
Lost          20
Name: count, dtype: int64
     CustID  Recency  Frequency   Monetary CustomerSegment
66       67       17         21  791769.41       Champions
124     125       79         24  720431.87           Loyal
18       19        9         24  708313.52       Champions
175     176       18         29  671459.13       Champions
117     118        5         19  671232.92       Champions


In [ ]:
master.to_csv('urbanmart_master.csv', index=False)
rfm.to_csv('urbanmart_customer_segments.csv', index=False)

with pd.ExcelWriter('urbanmart_dashboard.xlsx') as writer:
    monthly_rev.to_excel(writer, sheet_name='Monthly Trend')
    pivot_city_month.to_excel(writer, sheet_name='City x Quarter')
    rfm.to_excel(writer, sheet_name='Customer Segments', index=False)

total_rev = master['Revenue'].sum()
total_profit = master['Profit'].sum()
margin = total_profit / total_rev * 100
aov = master['Revenue'].mean()
guest_count = (master['cust_status'] == 'left_only').sum()

store_best = master.groupby(['StoreID', 'City'])['Revenue'].sum().sort_values(ascending=False)
best_store_id, best_store_city = store_best.index[0]
best_store_rev = store_best.iloc[0]

cat_best = master.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
month_best = monthly_rev['Revenue'].sort_values(ascending=False)

last3 = monthly_rev.tail(3)

seg_rev = (
    rfm.merge(known.groupby('CustID')['Revenue'].sum().rename('Rev'), on='CustID')
    .groupby('CustomerSegment')
    .agg(customers=('CustID', 'count'), revenue=('Rev', 'sum'))
    .reindex(['Champions', 'Loyal', 'At Risk', 'Lost'])
    .fillna(0)
)

region = master.groupby('Region')['Revenue'].sum()

line = '═' * 47
print(line)
print('     URBANMART — EXECUTIVE DASHBOARD REPORT')
print(line)
print('DATA PIPELINE SUMMARY')
print(f"  Raw Transactions   : {len(transactions):,}")
print(f"  After Cleaning     : {len(transactions_clean):,}  (removed {dupes_removed} dupes, {invalid_qty_removed} invalid qty)")
print(f"  Guest Purchases    : {guest_count:,} (CustID not in customer master)")
print()
print('REVENUE OVERVIEW')
print(f"  Total Revenue      : ₹{total_rev:,.0f}")
print(f"  Total Profit       : ₹{total_profit:,.0f}")
print(f"  Profit Margin      : {margin:.1f}%")
print(f"  Avg Order Value    : ₹{aov:,.0f}")
print()
print('TOP PERFORMERS')
print(f"  Best Store         : Store {best_store_id} in {best_store_city} (₹{best_store_rev:,.0f})")
print(f"  Best Category      : {cat_best.index[0]} (₹{cat_best.iloc[0]:,.0f})")
print(f"  Best Month         : {month_best.index[0]} (₹{month_best.iloc[0]:,.0f})")
print()
print('MONTHLY TREND (Last 3 Months)')
for m, row in last3.iterrows():
    print(f"  {m}: ₹{row['Revenue']:,.0f}  (MoM: {row['MoM_Growth']:.1f}%)")
print()
print('CUSTOMER SEGMENTS (RFM Analysis)')
for s, row in seg_rev.iterrows():
    print(f"  {s:<10} : {int(row['customers'])} customers (₹{row['revenue']:,.0f} revenue)")
print()
print('REGIONAL BREAKDOWN')
print(f"  North : ₹{region['North']:,.0f}   South : ₹{region['South']:,.0f}")
print(f"  East  : ₹{region['East']:,.0f}   West  : ₹{region['West']:,.0f}")
print()
print('FILES DELIVERED')
print('  ✅ urbanmart_master.csv')
print('  ✅ urbanmart_customer_segments.csv')
print('  ✅ urbanmart_dashboard.xlsx (3 sheets)')
print(line)

═══════════════════════════════════════════════
     URBANMART — EXECUTIVE DASHBOARD REPORT
═══════════════════════════════════════════════
DATA PIPELINE SUMMARY
  Raw Transactions   : 5,100
  After Cleaning     : 4,751  (removed 100 dupes, 249 invalid qty)
  Guest Purchases    : 214 (CustID not in customer master)

REVENUE OVERVIEW
  Total Revenue      : ₹118,881,918
  Total Profit       : ₹44,565,120
  Profit Margin      : 37.5%
  Avg Order Value    : ₹25,023

TOP PERFORMERS
  Best Store         : Store 7 in Kolkata (₹15,601,593)
  Best Category      : Home (₹48,328,288)
  Best Month         : 2023-05 (₹5,849,144)

MONTHLY TREND (Last 3 Months)
  2024-10: ₹4,987,905  (MoM: -13.5%)
  2024-11: ₹4,907,481  (MoM: -1.6%)
  2024-12: ₹4,433,928  (MoM: -9.6%)

CUSTOMER SEGMENTS (RFM Analysis)
  Champions  : 77 customers (₹39,682,615 revenue)
  Loyal      : 107 customers (₹42,419,321 revenue)
  At Risk    : 96 customers (₹26,609,961 revenue)
  Lost       : 20 customers (₹3,899,759 revenue)

R